# ASTD / ArSAS Replication — Arabic Sentiment Ensemble Paper
### Generalizability Beyond HARD

This notebook replicates the core HARD experiment (4 transformer backbones × ensemble
methods × multiple seeds) on two additional Arabic sentiment datasets — **ASTD**
(Nabil et al., 2015, ~10k tweets) and **ArSAS** (Elmadany et al., 2018, ~19.9k tweets)
— to populate the manuscript's "Generalizability Beyond HARD" table.

**Designed for Google Colab free-tier T4 GPUs.** Because free-tier sessions can
disconnect and have finite daily quota, this notebook:
1. Runs a **timing test first** (one model, one seed, one epoch) to measure real T4
   throughput before committing to the full run, and estimates total time.
2. **Checkpoints every result to Drive immediately** after each (dataset, model, seed)
   finishes, and **skips already-completed work on rerun** — a disconnect only costs
   you the one run in progress, not everything before it.
3. Keeps the exact same methodology as the original HARD experiment: same 4 backbones,
   same hyperparameters, same 4 ensemble strategies, same metrics — for a clean,
   comparable cross-dataset table.

**How to use:**
1. Run Sections 1–3 (setup, data acquisition, timing test) first.
2. Read the timing test's recommendation before running Section 4 (the full loop).
3. If your session disconnects, just re-run all cells — completed work is skipped
   automatically and it picks up where it left off.


## 1. Setup

In [1]:
# Install dependencies (safe to re-run)
!pip install -q transformers datasets torch scikit-learn accelerate


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
EL4ASA_ROOT = "/content/drive/MyDrive/EL4ASA"
REPLICATION_DIR = f"{EL4ASA_ROOT}/replication_astd_arsas"
CHECKPOINT_DIR = f"{REPLICATION_DIR}/results"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Results will be checkpointed to: {CHECKPOINT_DIR}")


Mounted at /content/drive
Results will be checkpointed to: /content/drive/MyDrive/EL4ASA/replication_astd_arsas/results


In [3]:
import re
import os
import json
import time
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup
)
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: No GPU detected -- go to Runtime > Change runtime type > T4 GPU before continuing.")


Device: cuda


## 2. Configuration

Same backbones, same hyperparameters, same seeds as the original HARD experiment,
for a directly comparable result. `MAX_LENGTH` is reduced from HARD's review-length
setting since tweets are much shorter.

In [4]:
MODELS = {
    "arabert": "aubmindlab/bert-base-arabert",
    "marbert": "UBC-NLP/MARBERT",
    "xlm-roberta": "xlm-roberta-base",
    "camelbert": "CAMeL-Lab/bert-base-arabic-camelbert-msa",
}

SEEDS = [42, 123, 456, 789, 2024]   # same as HARD -- reduce here if the timing test says to
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
NUM_EPOCHS = 3
MAX_LENGTH = 128          # tweets are short; HARD used up to 512 for full reviews
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.8, 0.1, 0.1
NUM_LABELS = 2            # binarized positive/negative, matching HARD

print("Config loaded.")
print(f"Models: {list(MODELS.keys())}")
print(f"Seeds: {SEEDS}")


Config loaded.
Models: ['arabert', 'marbert', 'xlm-roberta', 'camelbert']
Seeds: [42, 123, 456, 789, 2024]


## 3. Data acquisition and preprocessing

In [5]:
class ArabicPreprocessor:
    """Same preprocessing as the original HARD experiment, for consistency."""

    @staticmethod
    def normalize_arabic(text):
        if not isinstance(text, str):
            return ""
        text = re.sub("[\u0625\u0623\u0622\u0627]", "\u0627", text)  # alef variants -> alef
        text = re.sub("\u0649", "\u064a", text)  # alef maksura -> ya
        text = re.sub("\u0629", "\u0647", text)  # ta marbuta -> ha
        arabic_diacritics = re.compile("[\u0617-\u061A\u064B-\u0652\u0640]")
        text = re.sub(arabic_diacritics, "", text)
        return text

    @staticmethod
    def clean_text(text):
        if not isinstance(text, str):
            return ""
        text = re.sub(r"http\S+|www\S+|https\S+", "", text)
        text = re.sub(r"\S+@\S+", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    @staticmethod
    def preprocess(text):
        text = ArabicPreprocessor.normalize_arabic(text)
        text = ArabicPreprocessor.clean_text(text)
        return text


### 3a. One-time: download, preprocess, and save datasets to Drive

**Run this cell once.** It downloads ASTD and ArSAS, preprocesses and binarizes
them, and saves the results as CSV files in `EL4ASA/data/replication_astd_arsas_prepared/`
on Drive. Once those files exist, you can skip this cell on future runs (or delete
it entirely) — Section 3b below loads directly from Drive and will not
re-download anything.

Safe to re-run: if a prepared CSV already exists for a dataset, it's skipped.

In [6]:
DATA_CACHE_DIR = f"{EL4ASA_ROOT}/data/replication_astd_arsas_prepared"
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

ASTD_CACHE_PATH = f"{DATA_CACHE_DIR}/astd_prepared.csv"
ARSAS_CACHE_PATH = f"{DATA_CACHE_DIR}/arsas_prepared.csv"

ASTD_URL = "https://raw.githubusercontent.com/mahmoudnabil/ASTD/master/data/Tweets.txt"

def prepare_astd():
    df = pd.read_csv(ASTD_URL, sep="\t", header=None, names=["text", "raw_label"])
    label_map = {"POS": 1, "NEG": 0}  # drop OBJ and NEUTRAL/mixed
    df = df[df["raw_label"].isin(label_map.keys())].copy()
    df["label"] = df["raw_label"].map(label_map)
    df["text"] = df["text"].apply(ArabicPreprocessor.preprocess)
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    return df[["text", "label"]]


def prepare_arsas():
    from datasets import load_dataset
    ds = load_dataset("arbml/ArSAS", split="train")
    df = ds.to_pandas()

    assert "Tweet_text" in df.columns, (
        f"Expected column \'Tweet_text\' not found. Actual columns: {df.columns.tolist()}. "
        f"The ArSAS dataset schema may have changed -- inspect "
        f"https://huggingface.co/datasets/arbml/ArSAS and adjust this function\'s column names."
    )
    assert "label" in df.columns, f"Expected column \'label\' not found. Actual columns: {df.columns.tolist()}"

    label_names = ds.features["label"].names
    print(f"ArSAS raw label classes found: {label_names}")
    df["label_str"] = df["label"].apply(lambda i: label_names[i])

    label_map = {name: (1 if "positive" in name.lower() else 0)
                 for name in label_names if "positive" in name.lower() or "negative" in name.lower()}
    assert len(label_map) == 2, (
        f"Expected exactly one positive-like and one negative-like class name, got: {label_map}."
    )
    print(f"Binarization mapping: {label_map}")

    df = df[df["label_str"].isin(label_map.keys())].copy()
    df["label"] = df["label_str"].map(label_map)
    df["text"] = df["Tweet_text"].apply(ArabicPreprocessor.preprocess)
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    return df[["text", "label"]]


if os.path.exists(ASTD_CACHE_PATH):
    print(f"ASTD already prepared at {ASTD_CACHE_PATH} -- skipping download.")
else:
    print("Downloading and preparing ASTD...")
    astd_prepared = prepare_astd()
    astd_prepared.to_csv(ASTD_CACHE_PATH, index=False)
    print(f"Saved {len(astd_prepared)} samples to {ASTD_CACHE_PATH}")

if os.path.exists(ARSAS_CACHE_PATH):
    print(f"ArSAS already prepared at {ARSAS_CACHE_PATH} -- skipping download.")
else:
    print("Downloading and preparing ArSAS...")
    arsas_prepared = prepare_arsas()
    arsas_prepared.to_csv(ARSAS_CACHE_PATH, index=False)
    print(f"Saved {len(arsas_prepared)} samples to {ARSAS_CACHE_PATH}")

print("\nDone. You can skip or delete this cell on future runs -- Section 3b loads from Drive.")


ASTD already prepared at /content/drive/MyDrive/EL4ASA/data/replication_astd_arsas_prepared/astd_prepared.csv -- skipping download.
ArSAS already prepared at /content/drive/MyDrive/EL4ASA/data/replication_astd_arsas_prepared/arsas_prepared.csv -- skipping download.

Done. You can skip or delete this cell on future runs -- Section 3b loads from Drive.


### 3b. Load prepared datasets from Drive

This is the cell that actually runs every session. It loads the CSVs saved by
3a above — no downloading, no network dependency on GitHub/Hugging Face after
the first run.

In [7]:
DATA_CACHE_DIR = f"{EL4ASA_ROOT}/data/replication_astd_arsas_prepared"
ASTD_CACHE_PATH = f"{DATA_CACHE_DIR}/astd_prepared.csv"
ARSAS_CACHE_PATH = f"{DATA_CACHE_DIR}/arsas_prepared.csv"

for name, path in [("ASTD", ASTD_CACHE_PATH), ("ArSAS", ARSAS_CACHE_PATH)]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{name} prepared file not found at {path}. Run Section 3a above first "
            f"(it only needs to be run once, ever, per Drive account)."
        )

astd_df = pd.read_csv(ASTD_CACHE_PATH)
arsas_df = pd.read_csv(ARSAS_CACHE_PATH)

DATASETS = {
    "astd": astd_df,
    "arsas": arsas_df,
}
for name, df in DATASETS.items():
    print(f"{name}: {len(df)} total samples, {df["label"].mean()*100:.1f}% positive")


astd: 2419 total samples, 32.1% positive
arsas: 11784 total samples, 37.3% positive


## 4. Dataset splitting and PyTorch Dataset class

In [8]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def split_data(df, seed):
    train_val, test = train_test_split(
        df, test_size=TEST_RATIO, random_state=seed, stratify=df["label"]
    )
    val_ratio_adjusted = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    train, val = train_test_split(
        train_val, test_size=val_ratio_adjusted, random_state=seed, stratify=train_val["label"]
    )
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


## 5. Training and evaluation functions

In [9]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_one_model(model_name, model_id, train_df, val_df, seed, num_epochs=None, verbose=True):
    """Fine-tune one backbone on one seed's train split. Returns model, tokenizer, val acc, training time."""
    if num_epochs is None:
        num_epochs = NUM_EPOCHS
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=NUM_LABELS).to(DEVICE)

    train_ds = TextDataset(train_df["text"], train_df["label"], tokenizer, MAX_LENGTH)
    val_ds = TextDataset(val_df["text"], val_df["label"], tokenizer, MAX_LENGTH)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(WARMUP_RATIO * total_steps), num_training_steps=total_steps
    )

    best_val_acc = 0.0
    start_time = time.time()

    for epoch in range(num_epochs):
        model.train()
        loop = tqdm(train_loader, desc=f"{model_name} seed={seed} epoch={epoch+1}", disable=not verbose)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            if verbose:
                loop.set_postfix(loss=out.loss.item())

        val_acc, _, _, _ = evaluate_model(model, val_loader)
        if val_acc > best_val_acc:
            best_val_acc = val_acc

    training_time = time.time() - start_time
    return model, tokenizer, best_val_acc, training_time


def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            out = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(out.logits, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    return acc, np.array(all_preds), np.array(all_labels), np.array(all_probs)


def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1, "confusion_matrix": cm.tolist()}


def free_memory(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()


## 6. Timing test (run this before committing to the full loop)

Fine-tunes **one model** (CAMeLBERT, the fastest of the four in the original HARD
run per your `runtime_analysis` data) on **one seed** of **ASTD** (the smaller
dataset) for the full 3 epochs, times it, and extrapolates the total time for the
complete replication (4 models × 5 seeds × 2 datasets).

**Read the printed recommendation before running Section 7.**

In [10]:
print("Running timing test: camelbert, seed=42, ASTD...")
print("This trains for real (3 epochs) so the estimate is accurate, not a guess.\n")

_timing_train, _timing_val, _timing_test = split_data(DATASETS["astd"], seed=42)
_t0 = time.time()
_model, _tok, _val_acc, _train_time = train_one_model(
    "camelbert", MODELS["camelbert"], _timing_train, _timing_val, seed=42, verbose=True
)
_elapsed = time.time() - _t0
free_memory(_model, _tok)

print(f"\n--- Timing test result ---")
print(f"ASTD train size: {len(_timing_train)} samples")
print(f"Time for camelbert x 1 seed x ASTD (3 epochs): {_elapsed/60:.1f} minutes")
print(f"Validation accuracy reached: {_val_acc*100:.1f}%")

# Extrapolate: ArSAS is roughly 2x the size of ASTD, so scale accordingly.
_astd_size = len(DATASETS["astd"])
_arsas_size = len(DATASETS["arsas"])
_size_ratio = _arsas_size / _astd_size

_total_astd_runs = len(MODELS) * len(SEEDS)
_total_arsas_runs = len(MODELS) * len(SEEDS)
_est_astd_total = _elapsed * _total_astd_runs
_est_arsas_total = _elapsed * _size_ratio * _total_arsas_runs
_est_grand_total_hours = (_est_astd_total + _est_arsas_total) / 3600

print(f"\nEstimated ASTD total (4 models x {len(SEEDS)} seeds): {_est_astd_total/3600:.1f} hours")
print(f"Estimated ArSAS total (4 models x {len(SEEDS)} seeds, ~{_size_ratio:.1f}x ASTD size): {_est_arsas_total/3600:.1f} hours")
print(f"Estimated GRAND TOTAL: {_est_grand_total_hours:.1f} hours")
print(f"\n--- Recommendation ---")
if _est_grand_total_hours <= 3:
    print("Comfortably fits in a single Colab free session. Proceed with all 5 seeds.")
elif _est_grand_total_hours <= 8:
    print(f"Will need multiple sessions ({_est_grand_total_hours/3:.0f}+ sittings of ~3h).")
    print("This is fine -- checkpointing means you can stop and resume freely.")
    print("Proceeding with all 5 seeds is still reasonable; just expect several sessions.")
else:
    print(f"{_est_grand_total_hours:.0f} hours is a lot for free-tier T4 even with resuming.")
    print("Consider reducing SEEDS to e.g. [42, 123, 456] (3 seeds) in Section 2 above,")
    print("or running ASTD to completion first, then ArSAS in a separate pass on another day.")


Running timing test: camelbert, seed=42, ASTD...
This trains for real (3 epochs) so the estimate is accurate, not a guess.



config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/305k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

[transformers] BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSIN

camelbert seed=42 epoch=1:   0%|          | 0/121 [00:00<?, ?it/s]

camelbert seed=42 epoch=2:   0%|          | 0/121 [00:00<?, ?it/s]

camelbert seed=42 epoch=3:   0%|          | 0/121 [00:00<?, ?it/s]


--- Timing test result ---
ASTD train size: 1935 samples
Time for camelbert x 1 seed x ASTD (3 epochs): 2.5 minutes
Validation accuracy reached: 88.4%

Estimated ASTD total (4 models x 5 seeds): 0.8 hours
Estimated ArSAS total (4 models x 5 seeds, ~4.9x ASTD size): 4.0 hours
Estimated GRAND TOTAL: 4.9 hours

--- Recommendation ---
Will need multiple sessions (2+ sittings of ~3h).
This is fine -- checkpointing means you can stop and resume freely.
Proceeding with all 5 seeds is still reasonable; just expect several sessions.


## 7. Full training loop (checkpointed, resumable)

For each dataset, for each seed: fine-tune all 4 models, save each individual
model's result to Drive **immediately** after it finishes (not at the end), then
compute the 4 ensemble methods for that (dataset, seed) pair and save those too.

**Safe to stop and re-run at any point** — any (dataset, model, seed) combination
whose result file already exists on Drive is skipped.

In [11]:
def result_path(dataset_name, model_name, seed):
    return f"{CHECKPOINT_DIR}/{dataset_name}_{model_name}_seed{seed}.json"

def probs_path(dataset_name, model_name, seed):
    return f"{CHECKPOINT_DIR}/{dataset_name}_{model_name}_seed{seed}_probs.npz"

def ensemble_result_path(dataset_name, seed):
    return f"{CHECKPOINT_DIR}/{dataset_name}_ensembles_seed{seed}.json"

def already_done(path):
    return os.path.exists(path)

def save_json(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def save_probs(path, val_probs, val_labels, test_probs, test_labels, val_acc):
    np.savez(path, val_probs=val_probs, val_labels=val_labels,
              test_probs=test_probs, test_labels=test_labels, val_acc=val_acc)

def load_probs(path):
    d = np.load(path)
    return d["val_probs"], d["val_labels"], d["test_probs"], d["test_labels"], float(d["val_acc"])


In [12]:
# --- Ensemble methods (identical logic to the original HARD experiment) ---

def soft_voting(predictions_probs):
    avg_probs = np.mean(predictions_probs, axis=0)
    return np.argmax(avg_probs, axis=1), avg_probs

def hard_voting(predictions):
    stacked = np.column_stack(predictions)
    return np.array([np.argmax(np.bincount(row)) for row in stacked])

def weighted_voting(predictions_probs, weights):
    weights = np.array(weights, dtype=np.float64)
    weights = weights / np.sum(weights)
    weighted_probs = np.zeros_like(predictions_probs[0], dtype=np.float64)
    for i, probs in enumerate(predictions_probs):
        weighted_probs += weights[i] * np.array(probs, dtype=np.float64)
    return np.argmax(weighted_probs, axis=1), weighted_probs

def stacking_ensemble(val_probs, val_labels, test_probs, seed):
    X_val = np.hstack(val_probs)
    X_test = np.hstack(test_probs)
    meta_clf = LogisticRegression(max_iter=1000, random_state=seed)
    meta_clf.fit(X_val, val_labels)
    return meta_clf.predict(X_test), meta_clf.predict_proba(X_test)


In [13]:
def run_dataset_seed(dataset_name, df, seed):
    """Train all 4 models for one (dataset, seed) -- skipping any model whose
    probabilities are already checkpointed -- then compute all 4 ensembles.
    Genuinely resumable at per-model granularity: a disconnect only costs the
    one model that was mid-training, not the others already completed."""

    train_df, val_df, test_df = split_data(df, seed)

    val_probs_by_model = {}
    test_probs_by_model = {}
    val_metrics_by_model = {}
    val_labels_ref = None
    test_labels_ref = None

    for model_name, model_id in MODELS.items():
        rpath = result_path(dataset_name, model_name, seed)
        ppath = probs_path(dataset_name, model_name, seed)

        if already_done(ppath):
            print(f"  [{model_name}] already completed -- loading cached probabilities, skipping training.")
            val_probs, val_labels, test_probs, test_labels, best_val_acc = load_probs(ppath)
        else:
            model, tokenizer, best_val_acc, training_time = train_one_model(
                model_name, model_id, train_df, val_df, seed, verbose=True
            )

            val_ds = TextDataset(val_df["text"], val_df["label"], tokenizer, MAX_LENGTH)
            test_ds = TextDataset(test_df["text"], test_df["label"], tokenizer, MAX_LENGTH)
            val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
            test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

            _, val_preds, val_labels, val_probs = evaluate_model(model, val_loader)
            test_acc, test_preds, test_labels, test_probs = evaluate_model(model, test_loader)

            metrics = calculate_metrics(test_labels, test_preds)
            metrics["training_time"] = training_time
            metrics["val_accuracy"] = best_val_acc
            save_json(rpath, metrics)
            save_probs(ppath, val_probs, val_labels, test_probs, test_labels, best_val_acc)
            print(f"  [{model_name}] done, saved: {rpath}")

            free_memory(model, tokenizer)

        val_probs_by_model[model_name] = val_probs
        test_probs_by_model[model_name] = test_probs
        val_metrics_by_model[model_name] = best_val_acc
        val_labels_ref = val_labels
        test_labels_ref = test_labels

    # --- Ensembles for this (dataset, seed) -- only runs once all 4 models above are available ---
    epath = ensemble_result_path(dataset_name, seed)
    model_names = list(MODELS.keys())
    val_probs_list = [val_probs_by_model[m] for m in model_names]
    test_probs_list = [test_probs_by_model[m] for m in model_names]

    hard_preds = hard_voting([np.argmax(p, axis=1) for p in test_probs_list])
    soft_preds, _ = soft_voting(test_probs_list)
    weights = [val_metrics_by_model[m] for m in model_names]
    weighted_preds, _ = weighted_voting(test_probs_list, weights)
    stack_preds, _ = stacking_ensemble(val_probs_list, val_labels_ref, test_probs_list, seed)

    ensemble_results = {
        "hard_voting": calculate_metrics(test_labels_ref, hard_preds),
        "soft_voting": calculate_metrics(test_labels_ref, soft_preds),
        "weighted_voting": calculate_metrics(test_labels_ref, weighted_preds),
        "stacking": calculate_metrics(test_labels_ref, stack_preds),
    }
    save_json(epath, ensemble_results)
    print(f"  Saved ensembles: {epath}")


In [14]:
# --- Main loop ---
# Genuinely resumable: each model within each (dataset, seed) is checked
# individually via its cached probabilities file, and only missing ones are
# retrained. A disconnect costs at most the one model that was mid-training.

for dataset_name, df in DATASETS.items():
    separator = "=" * 70
    print(f"\n{separator}\nDATASET: {dataset_name}\n{separator}")
    for seed in SEEDS:
        epath = ensemble_result_path(dataset_name, seed)
        if already_done(epath):
            print(f"\n[{dataset_name} seed={seed}] Already fully completed -- skipping.")
            continue
        print(f"\n[{dataset_name} seed={seed}] Running (already-completed models within this seed will be skipped automatically)...")
        run_dataset_seed(dataset_name, df, seed)

print("\n\nAll requested (dataset, seed) combinations complete.")



DATASET: astd

[astd seed=42] Already fully completed -- skipping.

[astd seed=123] Already fully completed -- skipping.

[astd seed=456] Already fully completed -- skipping.

[astd seed=789] Already fully completed -- skipping.

[astd seed=2024] Already fully completed -- skipping.

DATASET: arsas

[arsas seed=42] Already fully completed -- skipping.

[arsas seed=123] Already fully completed -- skipping.

[arsas seed=456] Running (already-completed models within this seed will be skipped automatically)...
  [arabert] already completed -- loading cached probabilities, skipping training.
  [marbert] already completed -- loading cached probabilities, skipping training.


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

[transformers] BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSIN

camelbert seed=2024 epoch=1:   0%|          | 0/590 [00:00<?, ?it/s]

camelbert seed=2024 epoch=2:   0%|          | 0/590 [00:00<?, ?it/s]

camelbert seed=2024 epoch=3:   0%|          | 0/590 [00:00<?, ?it/s]

  [camelbert] done, saved: /content/drive/MyDrive/EL4ASA/replication_astd_arsas/results/arsas_camelbert_seed2024.json
  Saved ensembles: /content/drive/MyDrive/EL4ASA/replication_astd_arsas/results/arsas_ensembles_seed2024.json


All requested (dataset, seed) combinations complete.


## 8. Aggregate results

Combines all per-seed results into mean ± std, in the same shape as the
manuscript's Table 3/4 and the placeholder "Cross-Dataset Replication" table
in Section "Generalizability Beyond HARD".

In [15]:
def aggregate_dataset(dataset_name):
    rows = []
    for model_name in MODELS:
        f1s = []
        for seed in SEEDS:
            rpath = result_path(dataset_name, model_name, seed)
            if already_done(rpath):
                with open(rpath) as f:
                    f1s.append(json.load(f)["f1"] * 100)
        if f1s:
            rows.append({"model": model_name, "n_seeds": len(f1s),
                          "f1_mean": np.mean(f1s), "f1_std": np.std(f1s)})

    for ens_name in ["hard_voting", "soft_voting", "weighted_voting", "stacking"]:
        f1s = []
        for seed in SEEDS:
            epath = ensemble_result_path(dataset_name, seed)
            if already_done(epath):
                with open(epath) as f:
                    f1s.append(json.load(f)[ens_name]["f1"] * 100)
        if f1s:
            rows.append({"model": ens_name, "n_seeds": len(f1s),
                          "f1_mean": np.mean(f1s), "f1_std": np.std(f1s)})

    return pd.DataFrame(rows)


for dataset_name in DATASETS:
    print(f"\n=== {dataset_name.upper()} ===")
    summary = aggregate_dataset(dataset_name)
    if len(summary) == 0:
        print("No completed results yet.")
    else:
        summary["f1_formatted"] = summary.apply(lambda r: f"{r.f1_mean:.2f} ± {r.f1_std:.2f}", axis=1)
        display(summary[["model", "n_seeds", "f1_formatted"]])
        summary.to_csv(f"{CHECKPOINT_DIR}/{dataset_name}_summary.csv", index=False)
        print(f"Saved: {CHECKPOINT_DIR}/{dataset_name}_summary.csv")



=== ASTD ===


,model,n_seeds,f1_formatted
0,arabert,5,80.32 ± 2.96
1,marbert,5,87.62 ± 1.19
2,xlm-roberta,5,81.91 ± 1.94
3,camelbert,5,87.08 ± 1.67
4,hard_voting,5,86.26 ± 2.10
5,soft_voting,5,87.60 ± 1.28
6,weighted_voting,5,87.53 ± 1.30
7,stacking,5,87.13 ± 1.12


Saved: /content/drive/MyDrive/EL4ASA/replication_astd_arsas/results/astd_summary.csv

=== ARSAS ===


,model,n_seeds,f1_formatted
0,arabert,5,88.37 ± 0.52
1,marbert,5,91.24 ± 0.32
2,xlm-roberta,5,88.26 ± 0.58
3,camelbert,5,90.56 ± 0.68
4,hard_voting,5,91.34 ± 0.33
5,soft_voting,5,91.59 ± 0.14
6,weighted_voting,5,91.56 ± 0.18
7,stacking,5,91.51 ± 0.39


Saved: /content/drive/MyDrive/EL4ASA/replication_astd_arsas/results/arsas_summary.csv


## 9. LaTeX table row generator

Prints rows formatted for direct copy-paste into the manuscript's
"Cross-Dataset Replication" table (`tab:cross-dataset` in the .tex source),
replacing the `[TBD]` placeholders.

In [16]:
print("Copy these into the tab:cross-dataset table in the manuscript:\n")
for model_name in list(MODELS.keys()) + ["stacking"]:
    row = [model_name]
    for dataset_name in ["astd", "arsas"]:
        summary = aggregate_dataset(dataset_name)
        match = summary[summary["model"] == model_name]
        if len(match) > 0:
            r = match.iloc[0]
            row.append(f"{r.f1_mean:.2f} $\\pm$ {r.f1_std:.2f}")
        else:
            row.append("[TBD]")
    print(f"{row[0]} & {row[1]} & {row[2]} \\\\")


Copy these into the tab:cross-dataset table in the manuscript:

arabert & 80.32 $\pm$ 2.96 & 88.37 $\pm$ 0.52 \\
marbert & 87.62 $\pm$ 1.19 & 91.24 $\pm$ 0.32 \\
xlm-roberta & 81.91 $\pm$ 1.94 & 88.26 $\pm$ 0.58 \\
camelbert & 87.08 $\pm$ 1.67 & 90.56 $\pm$ 0.68 \\
stacking & 87.13 $\pm$ 1.12 & 91.51 $\pm$ 0.39 \\
